### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [2]:
import polars as pl
import pyarrow
import pandas as pd
import numpy as np

In [3]:
from pathlib import Path
folder= Path().resolve().parent /'input'


### 1. Load multiple dataset for different buildings.

In [4]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / 'load_profile_buildingID_*')

In [5]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [6]:
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
meta_data=(
    pl.scan_parquet(folder / 'MetaData.parquet')
)            
MetaData=meta_data.filter(
            pl.col('bldg_id').is_in(bldgs)).cast({"bldg_id": pl.UInt32}).select(
            pl.col('bldg_id', 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'))

# MetaData.row(0, named=True) 
MetaData.collect()

bldg_id,in.ashrae_iecc_climate_zone_2004_sub_cz_split
u32,str
10,"""1A - FL"""
100016,"""2A - FL, GA, AL, MS"""
100024,"""1A - FL"""


In [7]:
%%time
# merge the climate zone changes into the original data
df=data.join(MetaData, on='bldg_id')
df.collect_schema()

CPU times: user 2.5 ms, sys: 410 μs, total: 2.91 ms
Wall time: 1.82 ms


Schema([('bldg_id', UInt32),
        ('timestamp', Datetime(time_unit='ms', time_zone=None)),
        ('in.sqft', Float32),
        ('out.electricity.ceiling_fan.energy_consumption..kwh', Float32),
        ('out.electricity.clothes_dryer.energy_consumption..kwh', Float32),
        ('out.electricity.clothes_washer.energy_consumption..kwh', Float32),
        ('out.electricity.cooling.energy_consumption..kwh', Float32),
        ('out.electricity.cooling_fans_pumps.energy_consumption..kwh',
         Float32),
        ('out.electricity.dishwasher.energy_consumption..kwh', Float32),
        ('out.electricity.ev_charging.energy_consumption..kwh', Float32),
        ('out.electricity.freezer.energy_consumption..kwh', Float32),
        ('out.electricity.heating.energy_consumption..kwh', Float32),
        ('out.electricity.heating_fans_pumps.energy_consumption..kwh',
         Float32),
        ('out.electricity.heating_hp_bkup.energy_consumption..kwh', Float32),
        ('out.electricity.heating_

There are only time two climate zones for the data

----------------------------------------------

## Explore the data to get insights

In [8]:
import polars.selectors as pls
df=df.select(pls.matches(r'bldg_id|timestamp|in.ashrae_iecc_climate_zone_*|out.electricity.*?.energy_consumption..kwh'))

In [9]:
meta_data.collect()

bldg_id,upgrade,upgrade_name,weight,applicability,in.sqft,in.representative_income,in.county_name,in.ahs_region,in.aiannh_area,in.area_median_income,in.ashrae_iecc_climate_zone_2004,in.ashrae_iecc_climate_zone_2004_sub_cz_split,in.bathroom_spot_vent_hour,in.battery,in.bedrooms,in.building_america_climate_zone,in.cec_climate_zone,in.ceiling_fan,in.census_division,in.census_division_recs,in.census_region,in.city,in.clothes_dryer,in.clothes_dryer_usage_level,in.clothes_washer,in.clothes_washer_presence,in.clothes_washer_usage_level,in.cooking_range,in.cooking_range_usage_level,in.cooling_unavailable_days,in.cooling_setpoint,in.cooling_setpoint_has_offset,in.cooling_setpoint_offset_magnitude,in.cooling_setpoint_offset_period,in.corridor,in.county,…,out.emissions_reduction.fuel_oil.lrmer_high_re_cost_low_ng_price_15.co2e_kg,out.emissions_reduction.natural_gas.lrmer_high_re_cost_low_ng_price_15.co2e_kg,out.emissions_reduction.propane.lrmer_high_re_cost_low_ng_price_15.co2e_kg,out.emissions_reduction.all_fuels.lrmer_high_re_cost_low_ng_price_15.co2e_kg,out.emissions_reduction.electricity.aer_high_re_cost_avg.co2e_kg,out.emissions_reduction.fuel_oil.aer_high_re_cost_avg.co2e_kg,out.emissions_reduction.natural_gas.aer_high_re_cost_avg.co2e_kg,out.emissions_reduction.propane.aer_high_re_cost_avg.co2e_kg,out.emissions_reduction.all_fuels.aer_high_re_cost_avg.co2e_kg,out.emissions_reduction.electricity.aer_high_re_cost_low_ng_price_avg.co2e_kg,out.emissions_reduction.fuel_oil.aer_high_re_cost_low_ng_price_avg.co2e_kg,out.emissions_reduction.natural_gas.aer_high_re_cost_low_ng_price_avg.co2e_kg,out.emissions_reduction.propane.aer_high_re_cost_low_ng_price_avg.co2e_kg,out.emissions_reduction.all_fuels.aer_high_re_cost_low_ng_price_avg.co2e_kg,out.emissions_reduction.electricity.aer_low_re_cost_avg.co2e_kg,out.emissions_reduction.fuel_oil.aer_low_re_cost_avg.co2e_kg,out.emissions_reduction.natural_gas.aer_low_re_cost_avg.co2e_kg,out.emissions_reduction.propane.aer_low_re_cost_avg.co2e_kg,out.emissions_reduction.all_fuels.aer_low_re_cost_avg.co2e_kg,out.emissions_reduction.electricity.aer_low_re_cost_high_ng_price_avg.co2e_kg,out.emissions_reduction.fuel_oil.aer_low_re_cost_high_ng_price_avg.co2e_kg,out.emissions_reduction.natural_gas.aer_low_re_cost_high_ng_price_avg.co2e_kg,out.emissions_reduction.propane.aer_low_re_cost_high_ng_price_avg.co2e_kg,out.emissions_reduction.all_fuels.aer_low_re_cost_high_ng_price_avg.co2e_kg,out.emissions_reduction.electricity.aer_mid_case_avg.co2e_kg,out.emissions_reduction.fuel_oil.aer_mid_case_avg.co2e_kg,out.emissions_reduction.natural_gas.aer_mid_case_avg.co2e_kg,out.emissions_reduction.propane.aer_mid_case_avg.co2e_kg,out.emissions_reduction.all_fuels.aer_mid_case_avg.co2e_kg,out.bills.electricity.usd.savings,out.bills.fuel_oil.usd.savings,out.bills.natural_gas.usd.savings,out.bills.propane.usd.savings,out.bills.all_fuels.usd.savings,out.energy_burden.percentage.savings,upgrade.demand_flexibility_hvac,upgrade.demand_flexibility_random_shift
i64,i32,str,f64,bool,f64,f32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
1,28,"""HVAC Demand Flexibility - Pre-…",253.903673,true,1228.0,53672.0,"""Franklin County""","""Non-CBSA East South Central""","""No""","""80-100%""","""3A""","""3A""","""Hour0""","""None""","""4""","""Mixed-Humid""","""None""","""Standard Efficiency""","""East South Central""","""East South Central""","""South""","""Not in a census Place""","""Electric""","""100% Usage""","""Standard""","""Yes""","""100% Usage""","""Electric Resistance""","""100% Usage""","""Never""","""75F""","""Yes""","""2F""","""Night Setback -2h""","""Not Applicable""","""G0100590""",…,0.0,0.0,-6.304934,-6.885532,1.215628,0.0,0.0,-6.304934,-5.089306,1.174804,0.0,0.0,-6.304934,-5.13013,0.226796,0.0,0.0,-

## Preprocess the data

In [11]:
# defining the independant and dependant variables with encoding categorical variables and converting the datetime to a timestamp
x= df.select(pl.col("timestamp").dt.timestamp(), pl.col('out.electricity.total.energy_consumption..kwh').alias('Total Usage'),
             pl.col('in.ashrae_iecc_climate_zone_2004_sub_cz_split').cast(pl.Categorical).cast(pl.UInt32).alias('Climate Zone')).collect()
y=df.select(pl.col("out.electricity.cooling.energy_consumption..kwh").alias('Cooling Consumption'),
           pl.col("out.electricity.heating.energy_consumption..kwh").alias('Heating Consumption'),
           pl.col("out.electricity.freezer.energy_consumption..kwh").alias('Freezing Consumption'),
           pl.col("out.electricity.refrigerator.energy_consumption..kwh").alias('Refrigerator Consumption'),
           pl.col("out.electricity.dishwasher.energy_consumption..kwh").alias('Dishwasher Consumption')).collect()

## test standardizing the data with the standard scaler

In [ ]:
# from sklearn.preprocessing import StandardScaler
# scaler=StandardScaler()
# x=scaler.fit_transform(x)

## test standardizing the data with the Min Max Scaler

In [ ]:
# from sklearn.preprocessing import MinMaxScaler
# scaler=MinMaxScaler()
# x=scaler.fit_transform(x)

### Prepare cross-validation model


In [39]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    print(np.var(predicted))
    return mean_absolute_error(y_test, predicted)
        # r2_score(y_test, predicted)
        # mean_absolute_error(y_test, predicted), 
        # mean_squared_error(y_test, predicted), 
        # root_mean_squared_error(y_test, predicted)

In [40]:
%%time
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xg
skf= StratifiedKFold(n_splits=6)
kf=KFold(n_splits=10)
arr=[]
lf=pd.DataFrame()
for i, (train,test) in enumerate(kf.split(x,y)):
    x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
    LR=model(LinearRegression(), x_train, x_test, y_train, y_test)
    RF=model(RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1), x_train, x_test, y_train, y_test)
    XG=model(xg.XGBRegressor(), x_train, x_test, y_train, y_test)
    frame=pd.DataFrame({
    "Split Number": [i],
    "Linear Regression": [LR],
    "Random Forest": [RF],
    "XG Boost": [XG]
    })
    arr.append(frame)
lf=pd.concat(arr)

0.0018806597185544076
0.0011532428446227882
0.0010672022
0.0025893999072238182
0.004382588960259274
0.0021890434
0.004176325456547316
0.004896907258599078
0.002053468
0.0032644168535019033
0.006427220604040387
0.001611795
0.0019773456900077924
0.0049310795632493505
0.010738795
0.0025430926963733815
0.007190463698015772
0.02016055
0.004376109143620195
0.006409916506143038
0.0078929765
0.0023978636848175547
0.00196787716437077
0.001914812
0.0035704713701757975
0.0028737772779403995
0.0037898668
0.007329796457041308
0.0032790681620729362
0.002687787
CPU times: user 1min 52s, sys: 562 ms, total: 1min 53s
Wall time: 9.54 s


## Calculating measures

In [41]:
lf.mean()

Split Number         4.500000
Linear Regression    0.028079
Random Forest        0.022199
XG Boost             0.019776
dtype: float64

In [64]:
# prepare measurement
Test_data=x.head(1)
Actual_data=y.row(1)

In [66]:
Test_data

timestamp,Total Usage,Climate Zone
i64,f32,u32
1545212700000000,1.22827,1


In [64]:
Actual_data

(0.1280200034379959, 0.0, 0.0, 0.01245999988168478, 0.0)

In [65]:
# Select the best model 
import xgboost as xg
from sklearn.model_selection import train_test_split
x,x_test, y, y_test= train_test_split(x,y, test_size=.3, random_state=42)
xg_model=xg.XGBRegressor()
xg_model.fit(x,y)
estimated_values=xg_model.predict(Test_data)

In [68]:
estimated_values.flatten()

array([2.9516332e-02, 4.0768548e-03, 1.0928856e-17, 1.0773698e-02,
       2.7552096e-03], dtype=float32)

## Observation

Apparently using r square as a metric to represent to the data is a misrepresentation to the model's performance.
R square formula is - >1- ( total of squared residulas /total of squared standard deviation)
Knowing MAE is the total of residuals -> total(y - yPredicted)
giving r square = 0.02 therefore
giving standard deviation of .001 as given above
then r square= .0004 / .00001= 40 -> 1-40 =-39 which is wrong

Therefore, MAE score is the perfect representation of the data.
another testing metric should be provided.

Input data = timestamp Total      Usage     Climate Zone.
             1545212700000000     1.22827       1

Actual result=(0.1280200034379959, 0.0, 0.0, 0.01245999988168478, 0.0).


Estimated result= [2.9516332e-02, 4.0768548e-03, 1.0928856e-17, 1.0773698e-02,
       2.7552096e-03].